In [1]:
import pandas as pd
import json

In [2]:
# Load data
df = pd.read_csv("cases.csv")

# Drop rows with missing critical info
df = df.dropna(subset=["opname", "asa", "death_inhosp"])
df.head()

,caseid,subjectid,casestart,caseend,anestart,aneend,opstart,opend,adm,dis,...,intraop_colloid,intraop_ppf,intraop_mdz,intraop_ftn,intraop_rocu,intraop_vecu,intraop_eph,intraop_phe,intraop_epi,intraop_ca
0,1,5955,0,11542,-552,10848.0,1668,10368,-236220,627780,...,0,120,0.0,100,70,0,10,0,0,0
1,2,2487,0,15741,-1039,14921.0,1721,14621,-221160,1506840,...,0,150,0.0,0,100,0,20,0,0,0
2,3,2861,0,4394,-590,4210.0,1090,3010,-218640,40560,...,0,0,0.0,0,50,0,0,0,0,0
3,4,1903,0,20990,-778,20222.0,2522,17822,-201120,576480,...,0,80,0.0,100,100,0,50,0,0,0
4,5,4416,0,21531,-1009,22391.0,2591,20291,-67560,3734040,...,0,0,0.0,0,160,0,10,900,0,2100


In [ ]:
# grouped = df.groupby("opname").agg(
#     count=("caseid", "count"),
#     deaths=("death_inhosp", "mean"),
#     avg_asa=("asa", "mean")
# ).reset_index()

# grouped["commonality_score"] = 1 - grouped["count"] / grouped["count"].max()
# grouped["death_score"] = grouped["deaths"] / grouped["count"]
# grouped["asa_score"] = (grouped["avg_asa"] - 1) / 4  # scale ASA 1–5 to 0–1

# grouped["anxiety_score"] = (
#     0.4 * grouped["commonality_score"] +
#     0.3 * grouped["death_score"] +
#     0.3 * grouped["asa_score"]
# )



In [ ]:
# result = grouped[[
#     "opname", "count", "avg_asa", "commonality_score", "death_score", "asa_score", "anxiety_score"
# ]].to_dict(orient="records")

# with open("anxiety_scores.json", "w") as f:
#     json.dump(result, f, indent=2)

Kate's edits

In [30]:
grouped = df.groupby(["opname", "asa"]).agg(
    death_score=("death_inhosp", "mean"),
).reset_index()
grouped

,opname,asa,death_score
0,Abdominoperineal resection,1.0,0.0
1,Abdominoperineal resection,2.0,0.0
2,Abdominoperineal resection,3.0,0.0
3,Adhesiolysis,1.0,0.0
4,Adhesiolysis,2.0,0.0
...,...,...,...
478,Wound revision,3.0,0.0
479,anterior resection,1.0,0.0
480,subtotal colectomy,1.0,0.0
481,subtotal colectomy,2.0,0.0


In [31]:
result = grouped.merge(df, on=['opname', 'asa'])[['caseid', 'opname', 'asa', 'death_score']]
result = result.assign(asa_score = result['asa'])
result = result[['caseid', 'opname', 'asa_score', 'death_score']]
result.to_json("daniel.json", orient="records", lines=False)